In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.nn import *
class SwiGLU(Linear):
    def __init__(s, i, o=0, b=False, h=0):
        h = h or i
        o = o or h
        super().__init__(i, 2*(h or max(i, o)))
        s.bn = BatchNorm1d(i)
        s.ln = LayerNorm(i)
        s.ab = Linear(h or max(i, o), o or i, b)
        s.x = Linear(i, o) if o else Identity()
    def forward(s, x):
        a,b = super().forward(s.ln(s.bn(x))).chunk(2, 1)
        return s.ab(functional.silu(a)*b)+s.x(x)
m = Sequential(
    SwiGLU(1, 128),
    SwiGLU(128),
    SwiGLU(128),
    SwiGLU(128),
    SwiGLU(128),
    SwiGLU(128),
    SwiGLU(128, 1, True),
)
o = torch.optim.Adam(m.parameters(), .002, eps=1e-20)
O = torch.optim.lr_scheduler.CosineAnnealingLR(o, 200, .001)
x = torch.linspace(-100, 100, 1024)[:,None]
t = x.sinc()*(x/2+.1).sin()*(x/4+.2).sin()
for _ in range(10):
    for _ in range(20):
        y = m(x)
        fy = torch.fft.fft(y)
        l = functional.mse_loss(y, t)
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
    print(l)
plt.plot(t)
plt.plot(m.eval().requires_grad_(False)(x))

In [ ]:
a = torch.rand(1,16)
aa = GroupNorm(2,16,affine=False)(a)
aa.reshape(2,8).mean(1), aa.reshape(2,8).std(1)

In [ ]:
a-a_

In [ ]:
class Add(Sequential):
    def forward(s, x):
        return s[0](x)+s[1](x)
class F(Module):
    def __init__(s, f, *_):
        super().__init__()
        s.f = f
        s._ = _
    def forward(s, x):
        return s.f(x, *s._)
def SwiGLU(i, h=0, o=0, g=2, f=functional.silu):
    h = h or i
    return Add(
        Sequential(
            Linear(i, g*h),
            F(f),
            Unflatten(1, (g,-1)),
            F(torch.prod, 1),
            Linear(h, o or h, False),
        ),
        Linear(i, o or h, o),
    )
m = Sequential(
    SwiGLU(1, 16, f=torch.sin),
    SwiGLU(16, o=1, f=torch.sin),
)
o = torch.optim.Adam(m.parameters(), .1, eps=1e-9)
x = torch.linspace(-15, 15, 512)[:,None]
t = x.sinc()*(x/2+.1).sin()*(x/3+.2).sin()*(x/5+.4).sin()
for _ in range(10):
    for _ in range(100):
        l = functional.mse_loss(m(x), t)
        l.backward()
        o.step()
        o.zero_grad()
    print(l)
plt.plot(t)
plt.plot(m.eval().requires_grad_(False)(x))

In [ ]:
t